In [0]:
from pyspark.sql.functions import *

In [0]:
dbutils.fs.ls('abfss://maindevtest@mainstoragedev.dfs.core.windows.net/streamsrc/customerfull/')

In [0]:
customer=spark.read.format('csv').option('header','true').load('abfss://maindevtest@mainstoragedev.dfs.core.windows.net/streamsrc/customer/')
credit=spark.read.format('csv').option('header','true').load('abfss://maindevtest@mainstoragedev.dfs.core.windows.net/streamsrc/credit/')

In [0]:
customer_auto = (spark.readStream.format("cloudFiles")\
                 .option("cloudFiles.format", "csv") \
                 .option("cloudFiles.schemaLocation","/Volumes/maindatabricksdev/testdata/autoloader")\
                 .load("abfss://maindevtest@mainstoragedev.dfs.core.windows.net/streamsrc/customer")
                  )
customer_auto.writeStream.format("delta")\
 .option("checkpointLocation","abfss://maindevtest@mainstoragedev.dfs.core.windows.net/streamsrc/checkpoint")\
 .option("mergeSchema", "true")\
 .trigger(processingTime='5 seconds')\
 .outputMode("append")\
 .table("maindatabricksdev.testdata.customer_auto")

In [0]:
%%sql
select count(*) from maindatabricksdev.testdata.customer_auto